<a href="https://colab.research.google.com/github/tustus1022-ui/esaa/blob/main/1%EC%A3%BC%EC%B0%A8_%EC%A0%84%EC%B2%98%EB%A6%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install numerapi lightgbm

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
from numerapi import NumerAPI
napi = NumerAPI()

# v5.2에서 받을 수 있는 파일 목록 전체 확인
datasets = napi.list_datasets()
v52_files = [d for d in datasets if "v5.2" in d]
for f in v52_files:
    print(f)

v5.2/features.json
v5.2/live.parquet
v5.2/live_benchmark_models.parquet
v5.2/live_example_preds.csv
v5.2/live_example_preds.parquet
v5.2/meta_model.parquet
v5.2/train.parquet
v5.2/train_benchmark_models.parquet
v5.2/validation.parquet
v5.2/validation_benchmark_models.parquet
v5.2/validation_example_preds.csv
v5.2/validation_example_preds.parquet


In [6]:
import pandas as pd
import json

feature_metadata = json.load(open(f"/content/drive/MyDrive/{VERSION}/features.json"))
features = feature_metadata["feature_sets"]["small"]
columns = ["era"] + features + ["target"]

train = pd.read_parquet(f"/content/drive/MyDrive/{VERSION}/train.parquet", columns=columns)

# era 100개만 사용 (RAM 절약)
sample_eras = train["era"].unique()[:100]
train = train[train["era"].isin(sample_eras)]

print(train.shape)
print(train.head())

(351727, 44)
                   era  feature_antistrophic_striate_conscriptionist  \
id                                                                     
n0007b5abb0c3a25  0001                                             2   
n003bba8a98662e4  0001                                             2   
n003bee128c2fcfc  0001                                             2   
n0048ac83aff7194  0001                                             2   
n0055a2401ba6480  0001                                             2   

                  feature_bicameral_showery_wallaba  \
id                                                    
n0007b5abb0c3a25                                  2   
n003bba8a98662e4                                  2   
n003bee128c2fcfc                                  2   
n0048ac83aff7194                                  2   
n0055a2401ba6480                                  2   

                  feature_bridal_fingered_pensioner  \
id                                       

### **전처리 - feature engineering (lag 피처)**

lag 피처 생성

In [7]:
import numpy as np

def add_lag_features(df, features, lags=[1, 2, 4]):
    df = df.copy()

    # era 정렬
    eras = sorted(df["era"].unique())
    era_to_idx = {era: i for i, era in enumerate(eras)}
    df["era_idx"] = df["era"].map(era_to_idx)

    # era별 피처 평균 계산 후 shift
    era_means = df.groupby("era_idx")[features].mean()

    for lag in lags:
        lag_means = era_means.shift(lag)
        lag_means.columns = [f"{col}_lag{lag}" for col in features]
        df = df.merge(lag_means.reset_index(), on="era_idx", how="left")
        print(f"lag{lag} 피처 추가 완료")

    return df.drop(columns=["era_idx"])

train_lag = add_lag_features(train, features, lags=[1, 2, 4])
print(f"\n원본 피처 수: {len(features)}")
print(f"추가된 lag 피처 수: {train_lag.shape[1] - train.shape[1]}")
print(f"최종 shape: {train_lag.shape}")

lag1 피처 추가 완료
lag2 피처 추가 완료
lag4 피처 추가 완료

원본 피처 수: 42
추가된 lag 피처 수: 126
최종 shape: (351727, 170)


NaN 제거 및 모델 학습

In [8]:
from lightgbm import LGBMRegressor

# lag 없는 초반 era 제거
train_lag_clean = train_lag.dropna()
print(f"NaN 제거 후 shape: {train_lag_clean.shape}")

# 피처/타겟 분리
lag_cols = [c for c in train_lag_clean.columns if c not in ["era", "target"]]
X = train_lag_clean[lag_cols]
y = train_lag_clean["target"]

# era 기준 train/val 분리 (시간 순서 유지!)
eras = train_lag_clean["era"].unique()
split = int(len(eras) * 0.8)
train_eras, val_eras = eras[:split], eras[split:]

X_train = X[train_lag_clean["era"].isin(train_eras)]
y_train = y[train_lag_clean["era"].isin(train_eras)]
X_val   = X[train_lag_clean["era"].isin(val_eras)]
y_val   = y[train_lag_clean["era"].isin(val_eras)]

model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=5,
    colsample_bytree=0.1,
    random_state=42
)
model.fit(X_train, y_train)
print("학습 완료!")

NaN 제거 후 shape: (341913, 170)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.364522 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1680
[LightGBM] [Info] Number of data points in the train set: 261123, number of used features: 124
[LightGBM] [Info] Start training from score 0.499927
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fu

성능 평가

In [9]:
val_df = train_lag_clean[train_lag_clean["era"].isin(val_eras)].copy()
val_df["prediction"] = model.predict(X_val)

# era별 상관계수 (Numerai 리더보드 핵심 지표)
era_corrs = val_df.groupby("era").apply(
    lambda x: x["prediction"].corr(x["target"])
)

print(f"Era별 상관계수 평균: {era_corrs.mean():.5f}")
print(f"Era별 상관계수 표준편차: {era_corrs.std():.5f}")
print(f"샤프 비율 (mean/std): {era_corrs.mean() / era_corrs.std():.3f}")

Era별 상관계수 평균: 0.02142
Era별 상관계수 표준편차: 0.01304
샤프 비율 (mean/std): 1.643


/tmp/ipykernel_10260/4254761587.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  era_corrs = val_df.groupby("era").apply(
